<a href="https://colab.research.google.com/github/ascordero001-cell/enares-2024-crs04-ml/blob/main/04_ENARES_2024_STAGE1_reporte_cierre.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ENARES 2024 CRS04 ML Pipeline

## Notebook 4 — Stage 1 Closing Report

### Stage 1 — Data Ingestion

This notebook generates the official closing report for Stage 1 using the outputs previously produced by:

- Notebook 1 — Google Drive project structure creation;
- Notebook 2 — ENARES 2024 data ingestion from INEI;
- Notebook 3 — CRS01–CRS04 identification and CRS04 confirmation.

The purpose of this notebook is to consolidate technical evidence, ingestion metadata, structural validation results and reproducibility documentation into a single auditable markdown report.

---

# Notebook Information

| Field | Value |
|---|---|
| **Notebook name** | `04_ENARES_2024_STAGE1_reporte_cierre.ipynb` |
| **Stage** | Stage 1 — Data Ingestion |
| **Objective** | Generate the official Stage 1 closing report |
| **Role** | Computer Science Lead — Data Engineering |
| **Author** | Ana Cordero Ricaldi |
| **Account used** | `anacordero.001@gmail.com` |
| **Root Drive folder** | `/content/drive/MyDrive/ENARES_2024_PROJECT` |
| **Project scope** | ENARES 2024 CRS04 ML Pipeline |
| **Script version** | `stage1-closing-report-v1.0` |
| **Date executed** | `[AUTO-GENERATED AT RUNTIME]` |

---

# OBJECTIVE

Generate the final Stage 1 technical report documenting:

- Google Drive project structure creation;
- ingestion of official ENARES 2024 SPSS ZIP packages;
- extraction and preservation of raw `.sav` files;
- SHA-256 validation results;
- CRS01–CRS04 identification process;
- CRS04 confirmation evidence;
- generated manifests and logs;
- reproducibility guarantees;
- technical decisions adopted during Stage 1.

The resulting report serves as the official auditable summary of the ingestion stage.

---

# FUNCTION OF THIS NOTEBOOK

This notebook is responsible only for:

- reading outputs generated by Notebooks 1, 2 and 3;
- consolidating ingestion metadata;
- summarising Stage 1 execution results;
- documenting reproducibility evidence;
- generating the final markdown report;
- preparing the project for transition into Stage 2.

---

# INPUTS REQUIRED

```text
05Resultados/logs/
├── ENARES_2024_PROJECT_drive_folder_ids.csv
├── ENARES_2024_STAGE1_catalogo_modulos.csv
├── ENARES_2024_CRS04_variables_stage1.csv
├── ENARES_2024_CRS04_value_labels_stage1.csv
├── ENARES_2024_CRS04_missing_codes_stage1.csv
└── ENARES_2024_CRS04_validacion_stage1.csv

01BasesDatosPrimarias/
├── ENARES_2024_STAGE1_manifest_YYYYMMDD_HHMMSS.json
└── ENARES_2024_STAGE1_log_ingesta_YYYYMMDD_HHMMSS.txt

04CuestionariosInformes/reportes/
└── ENARES_2024_CRS_identificacion_modulos.md
```

---

# EXPECTED OUTPUT

```text
04CuestionariosInformes/reportes/
└── ENARES_2024_STAGE1_ingestion_report.md
```

---

# METHODOLOGICAL DECISION

The closing report is generated exclusively from outputs already produced during Stage 1.

This ensures:

- traceability of every ingestion action;
- reproducibility of technical decisions;
- auditability of generated metadata;
- preservation of provenance information;
- separation between raw data and processed outputs.

The report acts as the official technical evidence package for Stage 1 completion.

---

# STRICT LIMITS OF THIS NOTEBOOK

This notebook strictly **DOES NOT** perform:

- new downloads from INEI;
- modification of raw `.sav` files;
- data cleaning;
- recoding;
- statistical analysis;
- feature engineering;
- ML modelling;
- BigQuery loading.

Its sole purpose is documentation and Stage 1 closure reporting.

---

# REPRODUCIBILITY REQUIREMENT

The notebook must be executable using only the outputs generated during Stage 1 and must consistently reproduce:

- the Stage 1 summary report;
- ingestion statistics;
- module catalogues;
- CRS04 identification evidence;
- manifest references;
- validation summaries;
- reproducibility documentation.

All reported evidence must remain traceable to the original ENARES 2024 SPSS ZIP packages obtained from the official INEI Microdatos portal.

---

# CURRENT STATUS

Notebook prepared for:

- Stage 1 report consolidation;
- ingestion evidence summarisation;
- CRS04 confirmation documentation;
- validation review;
- reproducibility audit support;
- transition preparation for Stage 2.

In [1]:
import os
import json
import pandas as pd
from datetime import datetime

from google.colab import drive
drive.mount('/content/drive')

ROOT = "/content/drive/MyDrive/ENARES_2024_PROJECT"

RAW_DIR = os.path.join(ROOT, "01BasesDatosPrimarias")
LOG_DIR = os.path.join(ROOT, "05Resultados/logs")
REPORT_DIR = os.path.join(ROOT, "04CuestionariosInformes/reportes")

Mounted at /content/drive


In [2]:
# Ensure output directory exists
os.makedirs(REPORT_DIR, exist_ok=True)

# ---------------------------------------------------------
# 2. EXPLICIT INPUT VERIFICATION (Req 2)
# ---------------------------------------------------------
# Find latest manifest, log, and catalogue dynamically
def get_latest_file(directory, keyword):
    if not os.path.exists(directory): return None
    files = [f for f in os.listdir(directory) if keyword in f]
    return os.path.join(directory, sorted(files)[-1]) if files else None

manifest_path = get_latest_file(RAW_DIR, "manifest")
log_path = get_latest_file(RAW_DIR, "log_ingesta")
catalogue_path = get_latest_file(LOG_DIR, "catalogo")

required_files = {
    "Manifest JSON": manifest_path,
    "Log TXT": log_path,
    "Catalogue CSV": catalogue_path,
    "CRS04 Variables CSV": os.path.join(LOG_DIR, "ENARES_2024_CRS04_variables_stage1.csv"),
    "CRS04 Values CSV": os.path.join(LOG_DIR, "ENARES_2024_CRS04_value_labels_stage1.csv"),
    "CRS04 Validation CSV": os.path.join(LOG_DIR, "ENARES_2024_CRS04_validacion_stage1.csv"),
    "CRS04 Missing CSV": os.path.join(LOG_DIR, "ENARES_2024_CRS04_missing_codes_stage1.csv"),
    "CRS Ident Report": os.path.join(REPORT_DIR, "ENARES_2024_CRS_identificacion_modulos.md"),
    "Drive Folder IDs": os.path.join(LOG_DIR, "ENARES_2024_PROJECT_drive_folder_ids.csv")
}

input_status = []
inputs_missing = 0
inputs_read = 0

for name, path in required_files.items():
    if path and os.path.exists(path):
        size = os.path.getsize(path)
        status = "Exists"
        inputs_read += 1
    else:
        path = path if path else "Not Found in Directory"
        size = 0
        status = "Missing"
        inputs_missing += 1
    input_status.append({"Input": name, "Status": status, "Path": path, "Size(Bytes)": size})

inputs_df = pd.DataFrame(input_status)

# ---------------------------------------------------------
# 3. DATA EXTRACTION FOR REPORT
# ---------------------------------------------------------
# Manifest (Req 6 & 8)
manifest = []
manifest_rows = 0
modules_processed = 0
modules_failed = []
missing_modules = []
unexpected_modules = []
expected_modules = [f"976-Modulo{i}" for i in range(1941, 1963)]

if required_files["Manifest JSON"] and os.path.exists(required_files["Manifest JSON"]):
    with open(required_files["Manifest JSON"], "r") as f:
        manifest = json.load(f)
    manifest_rows = len(manifest)
    manifest_modules = [m.get("module_id", "") for m in manifest]
    missing_modules = sorted(set(expected_modules) - set(manifest_modules))
    unexpected_modules = sorted(set(manifest_modules) - set(expected_modules))
    modules_processed = len(manifest_modules)
    modules_failed = [m["module_id"] for m in manifest if m.get("status") != "success"]

# Log (Req 3 & 8)
log_lines_count = 0
log_errors = 0
log_timestamp = "Unknown"
if required_files["Log TXT"] and os.path.exists(required_files["Log TXT"]):
    with open(required_files["Log TXT"], "r") as f:
        lines = f.readlines()
        log_lines_count = len(lines)
        log_errors = sum(1 for line in lines if "ERROR" in line.upper() or "FAIL" in line.upper())
    log_timestamp = datetime.fromtimestamp(os.path.getmtime(required_files["Log TXT"])).strftime('%Y-%m-%d %H:%M:%S')

# Drive Structure (Req 5)
drive_folders = 0
main_folders_present = False
if os.path.exists(required_files["Drive Folder IDs"]):
    drive_df = pd.read_csv(required_files["Drive Folder IDs"])
    drive_folders = len(drive_df)
    folder_names = drive_df["folder_name"].dropna().tolist()
    required_folders = ["01BasesDatosPrimarias", "04CuestionariosInformes", "05Resultados", "99Codigos"]
    main_folders_present = all(any(req in fname for fname in folder_names) for req in required_folders)

# Catalogue (Req 8)
catalogue_rows = 0
file_types = {}
if required_files["Catalogue CSV"] and os.path.exists(required_files["Catalogue CSV"]):
    cat_df = pd.read_csv(required_files["Catalogue CSV"])
    catalogue_rows = len(cat_df)
    if "file_extension" in cat_df.columns:
        file_types = cat_df["file_extension"].value_counts().to_dict()

# CRS04 Validation (Req 9)
crs04_modules_info = []
total_raw_rows = 0
if os.path.exists(required_files["CRS04 Validation CSV"]):
    val_df = pd.read_csv(required_files["CRS04 Validation CSV"])
    for _, row in val_df.iterrows():
        crs04_modules_info.append({
            "module": row.get("module_id", "Unknown"),
            "file": row.get("file_name", "Unknown"),
            "rows": row.get("n_rows", 0),
            "cols": row.get("n_columns", 0)
        })
    total_raw_rows = val_df["n_rows"].sum()

# ---------------------------------------------------------
# 4. REPORT GENERATION
# ---------------------------------------------------------
report_path = os.path.join(REPORT_DIR, "ENARES_2024_STAGE1_ingestion_report.md")
current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

with open(report_path, "w", encoding="utf-8") as f:
    f.write("# ENARES 2024 - Stage 1 Ingestion Report\n\n")

    f.write("## Account and Objective\n")
    f.write("- **Account Used:** anacordero.001@gmail.com\n")
    f.write("- **Objective:** Reproducible ingestion of ENARES 2024 CRS04 dataset from INEI SPSS ZIP source, ensuring traceability, metadata preservation, and initial module identification.\n")
    f.write("- **Scope Notice:** Stage 1 documents ingestion and preservation only. No statistical analysis, data cleaning, recoding, merging, modeling, or BigQuery transfers have been performed at this stage.\n\n")

    f.write("## GitHub Repository\n")
    f.write("- **Repository:** https://github.com/ascordero001-cell/enares-2024-crs04-ml\n")
    f.write("- **Linked Issue:** Issue #7 - Write Stage 1 report\n")
    f.write("- **Linked Notebook:** `notebooks/01_ingesta/04_ENARES_2024_STAGE1_reporte_cierre.ipynb`\n")
    f.write("- **Linked Commit:** pending\n\n")

    f.write("## Official Data Source & Format\n")
    f.write("- **Source:** https://proyectos.inei.gob.pe/microdatos/\n")
    f.write("- **Raw Package:** Data downloaded as exact SPSS ZIP packages from INEI.\n")
    f.write("- **Format:** ZIP files preserved intact; `.sav` files are the confirmed raw analytical source. CSV/Stata formats were NOT used as primary sources to ensure metadata integrity.\n\n")

    f.write("## Drive Structure\n")
    f.write(f"- **Total Registered Folders:** {drive_folders}\n")
    f.write(f"- **Main Subfolders Present:** {main_folders_present} (01BasesDatosPrimarias, 04CuestionariosInformes, 05Resultados, 99Codigos)\n")
    f.write(f"- **Project Root Path:** `{ROOT}`\n\n")

    f.write("## Required Inputs Verification\n")
    f.write("| Input | Status | Size (Bytes) | Path |\n")
    f.write("|---|---|---|---|\n")
    for row in input_status:
        f.write(f"| {row['Input']} | {row['Status']} | {row['Size(Bytes)']} | `{row['Path']}` |\n")
    f.write("\n")

    f.write("## Modules Processed\n")
    f.write(f"- **Expected Modules (Total):** {len(expected_modules)} (Range: 976-Modulo1941 to 976-Modulo1962)\n")
    f.write(f"- **Processed Modules:** {modules_processed}\n")
    f.write(f"- **Failed Modules:** {len(modules_failed)}\n")
    if missing_modules:
        f.write(f"- **Missing Modules:** {', '.join(missing_modules)}\n")
    if unexpected_modules:
        f.write(f"- **Unexpected Modules:** {', '.join(unexpected_modules)}\n")
    f.write("\n")

    f.write("## Integrity Checks\n")
    f.write("- **ZIPs:** Downloaded and verified via SHA-256.\n")
    f.write("- **.sav Files:** Extracted and mapped in the catalogue.\n")
    f.write("- **PDFs:** Questionnaire and dictionary PDFs identified in catalogue.\n")
    f.write("- **Drive IDs:** Generated and stored securely.\n")
    f.write("- **Missing Codes:** Reviewed. If the missing codes output is empty, no SPSS user-missing metadata was exported/detected in the `.sav` headers.\n\n")

    f.write("## Manifest\n")
    f.write(f"- **File Used:** `{os.path.basename(str(manifest_path))}`\n")
    f.write(f"- **Total Records:** {manifest_rows}\n")
    f.write("- **Confirmation:** `selected_download_format = SPSS` and `official_selected_package_type = SPSS ZIP`.\n")
    f.write("- **Integrity:** SHA-256 checksums documented for all downloaded packages.\n\n")

    f.write("## Log\n")
    f.write(f"- **File Used:** `{os.path.basename(str(log_path))}`\n")
    f.write(f"- **Lines Processed:** {log_lines_count}\n")
    f.write(f"- **Inferred Date:** {log_timestamp}\n")
    f.write(f"- **Errors/Failures Detected:** {log_errors}\n\n")

    f.write("## Catalogue\n")
    f.write(f"- **File Used:** `{os.path.basename(str(catalogue_path))}`\n")
    f.write(f"- **Total Files Tracked:** {catalogue_rows}\n")
    f.write("- **Extensions Found:**\n")
    for ext, count in file_types.items():
        f.write(f"  - `{ext}`: {count} files\n")
    f.write("\n")

    f.write("## CRS01-CRS04 Identification\n")
    f.write("Based on the separate identification report (`ENARES_2024_CRS_identificacion_modulos.md`), CRS01, CRS02, CRS03, and CRS04 were successfully identified. CRS04 specifically corresponds to:\n")
    f.write("- `976-Modulo1959`\n")
    f.write("- `976-Modulo1960`\n")
    f.write("- `976-Modulo1961`\n")
    f.write("- `976-Modulo1962`\n")
    f.write("This was confirmed using evidence from filename patterns, SPSS metadata, variable labels, and questionnaire dimensions.\n\n")

    f.write("## CRS04 Initial Validation\n")
    f.write(f"**Total CRS04 raw rows across four files: {total_raw_rows}**\n")
    f.write("> **Note:** Rows are summed across CAP100, CAP200, CAP248, and CAP300. This is **not** yet the merged adolescent-level analytical sample.\n\n")
    f.write("| Module | SAV File | Rows | Columns |\n")
    f.write("|---|---|---|---|\n")
    for info in crs04_modules_info:
        f.write(f"| {info['module']} | {info['file']} | {info['rows']} | {info['cols']} |\n")
    f.write("\n")

    f.write("## Key CRS04 Candidate Variables Detected\n")
    f.write("- **Age & Sex:** Present in demographics.\n")
    f.write("- **Disability:** Candidates `C4P130_1` to `C4P130_6`.\n")
    f.write("- **Sample Weight:** `FACTOR_ALUMNOS`.\n")
    f.write("- **Strata:** `STRATA`.\n")
    f.write("- **Cluster/UPM:** `CCDD` (Department) and explicit school/ID identifiers to be verified in Stage 2.\n\n")

    f.write("## Issues Found\n")
    f.write("- No critical ingestion failures detected in manifest.\n")
    f.write("- No failed modules detected.\n")
    f.write("- Missing codes file was reviewed; if empty, no SPSS user-missing metadata were exported or detected.\n")
    f.write("- CRS04 row count is reported as raw rows across four files, not as a merged analytical sample.\n")
    if inputs_missing > 0:
        f.write(f"- **WARNING:** {inputs_missing} expected input file(s) were missing during report generation.\n")
    f.write("\n")

    f.write("## Pending Questions for Supervisor\n")
    f.write("1. Confirmation of CRS04 module boundaries (1959-1962).\n")
    f.write("2. Confirmation of interpretation regarding the CRS04 row sum vs. the final merged adolescent sample.\n")
    f.write("3. Confirmation of sample design variables: Weight (`FACTOR_ALUMNOS`), Strata (`STRATA`), and Cluster/UPM.\n")
    f.write("4. Validation of derived disability variable definition candidates (`C4P130_1`–`C4P130_6`).\n")
    f.write("5. Confirmation that Stage 2 must thoroughly validate merge keys before any joins occur.\n\n")

    f.write("## Output Summary\n")
    f.write(f"- **Report Generated At:** {current_time}\n")
    f.write(f"- **Report Path:** `{report_path}`\n")
    f.write(f"- **Inputs Read Successfully:** {inputs_read}\n")
    f.write(f"- **Inputs Missing:** {inputs_missing}\n")

    stage1_status = "Ready for Supervisor Review" if inputs_missing == 0 else "Requires Revision (Missing Inputs)"
    f.write(f"- **Stage 1 Status:** **{stage1_status}**\n")

print(f"Report successfully saved: {report_path}")
print(f"Inputs Found: {inputs_read} | Missing: {inputs_missing}")

Report successfully saved: /content/drive/MyDrive/ENARES_2024_PROJECT/04CuestionariosInformes/reportes/ENARES_2024_STAGE1_ingestion_report.md
Inputs Found: 9 | Missing: 0
